# 26 - Three dry initializations: compact recognition + paired uncertainty

**Python Colab + T4, Run All after25.** After reset run02 +04 on this SAME runtime. Requires completed13/14/23/24/25 and CPU audit12 artifacts on Drive. No training reruns. Notebook24 is read-only.

Input, original mixed L1 seed20261110, and ALL THREE dry-only seeds20261110/11/12; exactly original dry mild/moderate/severe plus identity. Reuse12 verified24 variants, compute8 new variants (20 total). Same full-image FP32 inference, original galleries and saved original-input NBIS thresholds. No calibration, checkpoint reselection, routing, other-family/alternate recognition or test access.

Report EACH dry seed and descriptive mean/sample SD/range. All dry seeds are compared to input and the SAME fixed mixed seed20261110 (not seed-matched new mixed models). This is not independent-data replication, best-seed selection, full stochastic uncertainty, isolated negative transfer or dose matching.

Reuse audit12's 2,000 paired probe-finger cluster draws within each DB, including all impressions/gallery attempts for a probe-finger. Same draws across conditions/methods/seeds; equal DB/three-dry-level weights. Pointwise95% conditional percentile intervals for FRR AND FAR, per seed. Also average paired draw arrays across the three FIXED dry checkpoints before computing the interval of their mean effect. This is NOT a confidence interval over a training-seed population. Seed SD is descriptive; never concatenate seed-repeated pairs as independent.

Intervals condition on fixed galleries, thresholds, selected checkpoints, simulator draws and these three initializations; exclude calibration/selection/training-seed uncertainty and multiplicity correction. Repeated development validation, not independent test, equivalence, real moisture or publication novelty. Report original-image recoveries/harms and FAR trade-offs; no automatic pass/fail decision or best-seed selection.

Atomic/checksummed completed variants resume; incomplete variants restart. Bootstrap reruns deterministically. Do not start duplicate execution while busy. Drive: `evaluations/dry_initialization_validation_v1/<hash>/`.

Completion: **DRY INITIALIZATION VALIDATION COMPLETE**. Send per-seed points, descriptive summary, paired intervals and original-image harms. Keep test sealed.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import linecache
import json
import math
import re
import shutil
import subprocess
import sys
import types
import zipfile
import io
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT, BOZORTH3 = (NBIS_ROOT / 'install' / 'bin' / name for name in ('mindtct', 'bozorth3'))
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel')
CONDITIONS = [('identity', 'none')] + [('dry_like', level) for level in ('mild', 'moderate', 'severe')]
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'alternative_corruption_validation_v1' / '476fc096ee2d51f03921'
REPLICATION_ROOT = PROJECT_ROOT / 'experiments' / 'm3_l1_initialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / '1374b21b3b10b3202a8d'
COMPARISONS = [('input', 'm0'), ('input', 'l1_control'), ('input', 'l1_sobel'), ('l1_control', 'l1_sobel')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact checksum mismatch: {directory / name}')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Evaluate compact dry-specialist diagnostic; no training.')
REPLICATION_ROOT = PROJECT_ROOT / 'experiments' / 'm6_dry_initialization_v1'
PRIOR24_DIR = PROJECT_ROOT / 'evaluations' / 'dry_specialization_validation_v1' / 'a2bcd5551cf4a97e3bf2'
AUDIT12_DIR = PROJECT_ROOT / 'analyses' / 'm1_validation_tradeoff_v1' / '851b89eb4be16b94389a'


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
validation = locked[locked['split'].eq('validation')].sort_values('relative_path').reset_index(drop=True)
if validation.empty or set(validation['dataset']) != {'FVC2000', 'FVC2002'} or validation.groupby(['dataset', 'db']).ngroups != 6:
    raise RuntimeError('Expected complete source-only validation in six databases.')
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m2_config = json.loads((M2_DIR / 'configuration.json').read_text(encoding='utf-8'))
m2_completion = json.loads((M2_DIR / 'completion.json').read_text(encoding='utf-8'))
reference_config = json.loads((REFERENCE_DIR / 'configuration.json').read_text(encoding='utf-8'))
reference_completion = json.loads((REFERENCE_DIR / 'completion.json').read_text(encoding='utf-8'))
m0_best = json.loads((M0_DIR / 'best.json').read_text(encoding='utf-8'))
m0_completion = json.loads((M0_DIR / 'completion.json').read_text(encoding='utf-8'))
m0_sha, m2_sha, reference_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M2_DIR, REFERENCE_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M2_DIR, m2_config['source_file_sha256'])
verify_files(M2_DIR, m2_completion['file_sha256'])
verify_files(REFERENCE_DIR, reference_completion['file_sha256'])
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (pair_config, sim_config, m0_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (file_digest(PAIR_DIR / 'pairs.csv') != pair_config['pair_csv_sha256']
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m2_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen pairs/simulator provenance mismatch.')
if (m0_completion['status'] != 'm0_training_complete' or m0_completion['configuration_sha256'] != m0_sha
        or m0_completion['epochs_completed'] != m0_config['epochs'] or m0_completion['best_checkpoint'] != m0_best
        or m0_best['configuration_sha256'] != m0_sha or file_digest(M0_DIR / m0_best['file']) != m0_best['sha256']):
    raise RuntimeError('M0 completion/checkpoint provenance mismatch.')
if (m2_completion['status'] != 'structure_loss_control_complete' or m2_completion['configuration_sha256'] != m2_sha
        or m2_config['m0_configuration_sha256'] != m0_sha or set(m2_config['run_names']) != {'l1_control', 'l1_sobel'}):
    raise RuntimeError('M2 completed loss experiment provenance mismatch.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M2_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M2 model/source differs from M0: {name}')
if (reference_completion['status'] != 'm1_controlled_validation_matching_complete'
        or reference_completion['configuration_sha256'] != reference_sha
        or reference_config['m0_configuration_sha256'] != m0_sha
        or reference_config['model_checkpoints']['m0'] != m0_best
        or reference_config['pair_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or reference_config['pair_csv_sha256'] != pair_config['pair_csv_sha256']
        or reference_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')):
    raise RuntimeError('Notebook-11 reference provenance mismatch.')
helper_path = REFERENCE_DIR / 'frozen_nb08_helpers.py.txt'
if file_digest(helper_path) != reference_config['frozen_helpers_sha256']:
    raise RuntimeError('Frozen NBIS/inference/metric helper source changed.')
HELPER_SOURCE = helper_path.read_bytes().decode('utf-8')
helper_code = compile(HELPER_SOURCE, str(helper_path), 'exec')
for config in (m0_config, m2_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or Image.__version__ != config['pillow_version'] or np.__version__ != config['numpy_version']):
        raise RuntimeError('Runtime library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
if not (NBIS_ROOT / 'verified_build.json').is_file():
    raise FileNotFoundError('NBIS runtime installation missing. Run notebook 04 first on this runtime.')
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if (build['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['nbis_source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
        or pair_config['mindtct_flags'] != [] or pair_config['bozorth3_flags'] != ['-A', 'minminutiae=10']):
    raise RuntimeError('NBIS source/settings differ from the frozen pair protocol.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError(f'NBIS binary missing/modified: {binary}')
    subprocess.run([str(binary), '-version'], capture_output=True, text=True, check=True)
pairs = pd.read_csv(PAIR_DIR / 'pairs.csv', keep_default_na=False)
if pairs['pair_id'].duplicated().any() or validation['relative_path'].duplicated().any() or pairs['duplicate_sensitivity_eligible'].dtype != bool:
    raise RuntimeError('Invalid pair/image keys or duplicate-sensitivity encoding.')
probe_records = validation[validation['impression_id'].ne(1)].to_dict('records')
gallery_records = validation[validation['impression_id'].eq(1)].to_dict('records')
if (set(pairs['probe_path']) != {row['relative_path'] for row in probe_records}
        or set(pairs['gallery_path']) != {row['relative_path'] for row in gallery_records}):
    raise RuntimeError('Frozen pairs do not cover all validation probes/galleries.')
for record in validation.to_dict('records'):
    path = DATA_ROOT / record['relative_path']
    if not path.is_file() or file_digest(path) != record['sha256']:
        raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02 on this runtime.')
training = load_module('m2_matching_model_v1', M2_DIR / 'restoration_baseline.py')
simulator = load_module('m2_matching_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
models, model_pointers = {}, {'m0': m0_best}
for name in ('l1_control', 'l1_sobel'):
    directory = M2_DIR / name
    if file_digest(directory / 'completion.json') != m2_completion['run_completion_sha256'][name]:
        raise RuntimeError(f'M2 run completion changed: {name}')
    marker = json.loads((directory / 'completion.json').read_text(encoding='utf-8'))
    verify_files(directory, marker['file_sha256'])
    pointer = json.loads((directory / 'best.json').read_text(encoding='utf-8'))
    if (marker['configuration_sha256'] != m2_sha or marker['run'] != name
            or marker['epochs_completed'] != len(m2_config['epoch_conditions'])
            or marker['result']['best_checkpoint'] != pointer or pointer['configuration_sha256'] != m2_sha
            or file_digest(directory / pointer['file']) != pointer['sha256']):
        raise RuntimeError(f'M2 best checkpoint/completion mismatch: {name}')
    model_pointers[name] = pointer
for name in ('m0', 'l1_control', 'l1_sobel'):
    directory = M0_DIR if name == 'm0' else M2_DIR / name
    state = torch.load(directory / model_pointers[name]['file'], map_location='cpu', weights_only=True)
    expected_sha = m0_sha if name == 'm0' else m2_sha
    if state['configuration_sha256'] != expected_sha:
        raise RuntimeError(f'Model payload configuration mismatch: {name}')
    if (name == 'm0' and state['epoch'] != m0_best['epoch']) or (name != 'm0' and
        (state['run'] != name or state['completed_epochs'] != model_pointers[name]['completed_epochs'])):
        raise RuntimeError(f'Model payload checkpoint selection mismatch: {name}')
    model = training.RestorationUNet(base=m0_config['base_channels']).to(DEVICE)
    model.load_state_dict(state['model'], strict=True)
    model.eval()
    models[name] = model
del state
prior_config = json.loads((PRIOR_DIR / 'configuration.json').read_text(encoding='utf-8'))
prior_completion = json.loads((PRIOR_DIR / 'completion.json').read_text(encoding='utf-8'))
prior_sha = file_digest(PRIOR_DIR / 'configuration.json')
if (prior_completion['status'] != 'structure_validation_matching_complete'
        or prior_completion['configuration_sha256'] != prior_sha or prior_completion['variants_available'] != 40
        or set(prior_completion['methods']) != set(METHODS)):
    raise RuntimeError('Expected completed notebook14 with the same four fixed methods.')
verify_files(PRIOR_DIR, prior_completion['file_sha256'])
if (prior_config['model_checkpoints'] != model_pointers
        or prior_config['m0_configuration_sha256'] != m0_sha or prior_config['m2_configuration_sha256'] != m2_sha
        or prior_config['pair_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or prior_config['pair_csv_sha256'] != file_digest(PAIR_DIR / 'pairs.csv')
        or prior_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or prior_config['frozen_helpers_sha256'] != file_digest(helper_path)):
    raise RuntimeError('Checkpoint, normalization, helper or pair provenance differs from notebook14.')
current_reference_settings = {'nbis_binary_sha256': build['binary_sha256'], 'torch_version': str(torch.__version__),
    'torch_cuda': str(torch.version.cuda), 'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'inference': 'one full native normalized image, FP32, no AMP, no padding, no tiles',
    'quantization': 'numpy.rint(image * 255) to uint8; lossless PNG at 500dpi', 'tf32_enabled': False}
reference_mismatches = [name for name, value in current_reference_settings.items()
                        if prior_config['inference_and_matcher'].get(name) != value]
if reference_mismatches:
    raise RuntimeError(f'Notebook14 runtime/matcher differs: {reference_mismatches}. Restore its runtime/build or rerun notebook14 on this runtime first; never mix scores.')
saved_thresholds = pd.read_csv(PRIOR_DIR / 'validation_thresholds.csv')
if saved_thresholds.duplicated(['condition', 'method', 'dataset', 'db']).any():
    raise RuntimeError('Duplicate frozen threshold rows.')
threshold_reference = saved_thresholds[saved_thresholds['condition'].eq('identity/none') & saved_thresholds['method'].eq('input')]
fixed_thresholds = {(row['dataset'], row['db']): row['original_fixed_threshold'] for row in threshold_reference.to_dict('records')}
if set(fixed_thresholds) != set(zip(validation['dataset'], validation['db'])):
    raise RuntimeError('Missing source-database thresholds.')
for row in saved_thresholds.to_dict('records'):
    value = fixed_thresholds.get((row['dataset'], row['db']))
    if value is None or value != row['original_fixed_threshold'] or not np.isfinite(value) or value < 0 or value != int(value):
        raise RuntimeError('Original-input thresholds are not fixed finite nonnegative integers.')
print('Locked probes:', len(probe_records), '| Pairs per variant:', len(pairs))
print('The 12 diagnostic variants will use current-build scores, original galleries and saved notebook14 thresholds.')


In [ ]:
specialist_pointer_path = SPECIALIZATION_ROOT / 'completed_experiment.json'
if not specialist_pointer_path.is_file():
    raise FileNotFoundError('Notebook23 has not published a completed run. Complete notebook23 first.')
specialist_pointer = json.loads(specialist_pointer_path.read_text(encoding='utf-8'))
SPECIALIZATION_DIR = Path(specialist_pointer['directory'])
if (SPECIALIZATION_DIR.parent != SPECIALIZATION_ROOT
        or file_digest(SPECIALIZATION_DIR / 'configuration.json') != specialist_pointer['configuration_sha256']
        or file_digest(SPECIALIZATION_DIR / 'completion.json') != specialist_pointer['completion_sha256']):
    raise RuntimeError('Notebook23 published pointer/integrity mismatch.')
specialist_config = json.loads((SPECIALIZATION_DIR / 'configuration.json').read_text(encoding='utf-8'))
specialist_completion = json.loads((SPECIALIZATION_DIR / 'completion.json').read_text(encoding='utf-8'))
specialist_sha = specialist_pointer['configuration_sha256']
expected_runs = {'dry_l1_seed20261110': 20261110}
expected_conditions = [[(family, severity) if family == 'identity' else ('dry_like', severity)
    for family, severity in conditions] for conditions in m2_config['epoch_conditions']]
if (specialist_completion['status'] != 'dry_specialization_complete'
        or specialist_completion['configuration_sha256'] != specialist_sha
        or specialist_config['version'] != 'm5_dry_specialization_v1' or specialist_config['seed'] != 20261110
        or specialist_config['initialization_seeds'] != expected_runs
        or set(specialist_config['run_names']) != set(expected_runs)
        or set(specialist_completion['run_completion_sha256']) != set(expected_runs)
        or specialist_config['m2_configuration_sha256'] != m2_sha
        or specialist_config['m2_completion_sha256'] != file_digest(M2_DIR / 'completion.json')
        or specialist_config['m0_configuration_sha256'] != m0_sha
        or specialist_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or specialist_config['settings'] != m2_config['settings']
        or specialist_config['original_epoch_conditions'] != m2_config['epoch_conditions']
        or specialist_config['epoch_conditions'] != json.loads(json.dumps(expected_conditions))
        or specialist_config['original_l1_best_checkpoint'] != model_pointers['l1_control']
        or specialist_config['gradient_lambda'] != 0
        or specialist_config['losses'] != {'dry_l1_seed20261110': 'global L1'}
        or specialist_config['explicit_identity_fraction'] != m2_config['explicit_identity_fraction']
        or specialist_config['explicit_identity_per_source_epoch'] != m2_config['explicit_identity_per_source_epoch']
        or specialist_config['tasks_per_run'] != m2_config['tasks_per_run']
        or specialist_config['updates_per_run'] != m2_config['updates_per_run']
        or specialist_config['alternative_protocol_used_for_training_or_selection']
        or specialist_config['dry_checkpoint_reselection']
        or specialist_config['selection_metric'] != 'same fixed validation mean global L1; initial identity checkpoint included'):
    raise RuntimeError('Notebook23 dry-specialization recipe/completion differs.')
verify_files(SPECIALIZATION_DIR, specialist_completion['file_sha256'])
verify_files(SPECIALIZATION_DIR, specialist_config['source_file_sha256'])
for filename in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv', 'curriculum_data.py',
                 'preservation_data.py', 'frozen_m1_engine.py.txt', 'order_shared.csv', 'identity_replacement_ids.csv'):
    if file_digest(SPECIALIZATION_DIR / filename) != m2_config['source_file_sha256'][filename]:
        raise RuntimeError(f'Specialist source/order/identity artifact changed: {filename}')
for key in ('torch_version', 'torch_cuda', 'numpy_version', 'pillow_version', 'gpu_name', 'gpu_compute_capability'):
    if specialist_config[key] != current_reference_settings[key]:
        raise RuntimeError(f'Specialist training/inference runtime differs: {key}')
name = 'dry_l1_seed20261110'
directory = SPECIALIZATION_DIR / name
if file_digest(directory / 'completion.json') != specialist_completion['run_completion_sha256'][name]:
    raise RuntimeError('Specialist run completion changed.')
marker = json.loads((directory / 'completion.json').read_text(encoding='utf-8'))
verify_files(directory, marker['file_sha256'])
pointer = json.loads((directory / 'best.json').read_text(encoding='utf-8'))
if (marker['configuration_sha256'] != specialist_sha or marker['run'] != name
        or marker['epochs_completed'] != len(m2_config['epoch_conditions'])
        or marker['result']['initialization_seed'] != 20261110
        or marker['result']['best_checkpoint'] != pointer or pointer['configuration_sha256'] != specialist_sha
        or file_digest(directory / pointer['file']) != pointer['sha256']):
    raise RuntimeError('Specialist selected checkpoint mismatch.')
state = torch.load(directory / pointer['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != specialist_sha or state['run'] != name
        or state['initialization_seed'] != 20261110 or state['completed_epochs'] != pointer['completed_epochs']):
    raise RuntimeError('Specialist checkpoint payload differs.')
specialist_model = training.RestorationUNet(base=m0_config['base_channels']).to(DEVICE)
specialist_model.load_state_dict(state['model'], strict=True)
specialist_model.eval()
models = {'l1_seed20261110': models['l1_control'], name: specialist_model}
model_pointers = {'l1_seed20261110': model_pointers['l1_control'], name: pointer}
METHODS = ('input', 'l1_seed20261110', 'dry_l1_seed20261110')
COMPARISONS = [('input', 'l1_seed20261110'), ('input', 'dry_l1_seed20261110'),
    ('l1_seed20261110', 'dry_l1_seed20261110')]
del state
print('Loaded one fixed dry-specialist checkpoint, epoch:', pointer['completed_epochs'])
print('Four conditions only; no routing, dry reselection, calibration or alternate/test access.')


In [ ]:
replication_pointer = json.loads((REPLICATION_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
REPLICATION_DIR = Path(replication_pointer['directory'])
if (REPLICATION_DIR.parent != REPLICATION_ROOT
        or file_digest(REPLICATION_DIR / 'configuration.json') != replication_pointer['configuration_sha256']
        or file_digest(REPLICATION_DIR / 'completion.json') != replication_pointer['completion_sha256']):
    raise RuntimeError('Notebook25 pointer/integrity mismatch.')
replication_config = json.loads((REPLICATION_DIR / 'configuration.json').read_text(encoding='utf-8'))
replication_completion = json.loads((REPLICATION_DIR / 'completion.json').read_text(encoding='utf-8'))
replication_sha = replication_pointer['configuration_sha256']
new_seeds = {'dry_l1_seed20261111': 20261111, 'dry_l1_seed20261112': 20261112}
if (replication_completion['status'] != 'dry_initialization_replication_complete'
        or replication_completion['configuration_sha256'] != replication_sha
        or replication_config['version'] != 'm6_dry_initialization_v1'
        or replication_config['initialization_seeds'] != new_seeds
        or set(replication_config['run_names']) != set(new_seeds)
        or replication_config['original_dry_configuration_sha256'] != specialist_sha
        or replication_config['original_dry_completion_sha256'] != file_digest(SPECIALIZATION_DIR / 'completion.json')
        or replication_config['original_dry_best_checkpoint'] != model_pointers['dry_l1_seed20261110']):
    raise RuntimeError('Notebook25 original dry provenance/recipe mismatch.')
verify_files(REPLICATION_DIR, replication_completion['file_sha256'])
verify_files(REPLICATION_DIR, replication_config['source_file_sha256'])
for key in ('seed', 'settings', 'epoch_conditions', 'original_epoch_conditions', 'm2_configuration_sha256',
        'm2_completion_sha256', 'gradient_lambda', 'selection_metric', 'explicit_identity_fraction',
        'explicit_identity_per_source_epoch', 'tasks_per_run', 'updates_per_run', 'task_mapping',
        'alternative_protocol_used_for_training_or_selection', 'dry_checkpoint_reselection',
        'torch_version', 'torch_cuda', 'numpy_version', 'pillow_version', 'gpu_name', 'gpu_compute_capability'):
    if replication_config[key] != specialist_config[key]:
        raise RuntimeError(f'Dry initialization changed fixed recipe/runtime: {key}')
for filename, digest in specialist_config['source_file_sha256'].items():
    if replication_config['source_file_sha256'].get(filename) != digest:
        raise RuntimeError(f'Dry initialization changed frozen source: {filename}')
if replication_config['losses'] != {name: 'global L1' for name in new_seeds}:
    raise RuntimeError('Dry initialization objective differs.')
for name, seed in new_seeds.items():
    directory = REPLICATION_DIR / name
    if file_digest(directory / 'completion.json') != replication_completion['run_completion_sha256'][name]:
        raise RuntimeError(f'Dry replicated completion changed: {name}')
    marker = json.loads((directory / 'completion.json').read_text(encoding='utf-8'))
    verify_files(directory, marker['file_sha256'])
    pointer = json.loads((directory / 'best.json').read_text(encoding='utf-8'))
    if (marker['configuration_sha256'] != replication_sha or marker['run'] != name
            or marker['epochs_completed'] != len(specialist_config['epoch_conditions'])
            or marker['result']['initialization_seed'] != seed
            or marker['result']['best_checkpoint'] != pointer
            or pointer['configuration_sha256'] != replication_sha
            or file_digest(directory / pointer['file']) != pointer['sha256']):
        raise RuntimeError(f'Dry replicated selected checkpoint differs: {name}')
    state = torch.load(directory / pointer['file'], map_location='cpu', weights_only=True)
    if (state['configuration_sha256'] != replication_sha or state['run'] != name
            or state['initialization_seed'] != seed or state['completed_epochs'] != pointer['completed_epochs']):
        raise RuntimeError(f'Dry replicated payload mismatch: {name}')
    model = training.RestorationUNet(base=m0_config['base_channels']).to(DEVICE)
    model.load_state_dict(state['model'], strict=True)
    models[name], model_pointers[name] = model.eval(), pointer
METHODS = ('input', 'l1_seed20261110', 'dry_l1_seed20261110', *new_seeds)
DRY_METHODS = METHODS[2:]
COMPARISONS = [('input', 'l1_seed20261110')] + [(baseline, method)
    for baseline in ('input', 'l1_seed20261110') for method in DRY_METHODS]
prior24_config = json.loads((PRIOR24_DIR / 'configuration.json').read_text(encoding='utf-8'))
prior24_completion = json.loads((PRIOR24_DIR / 'completion.json').read_text(encoding='utf-8'))
prior24_sha = file_digest(PRIOR24_DIR / 'configuration.json')
if (prior24_completion['status'] != 'dry_specialization_validation_complete'
        or prior24_completion['configuration_sha256'] != prior24_sha
        or prior24_completion['variants_available'] != 12
        or prior24_config['specialist_configuration_sha256'] != specialist_sha
        or prior24_config['specialist_completion_sha256'] != file_digest(SPECIALIZATION_DIR / 'completion.json')
        or prior24_config['reference_evaluation_sha256'] != prior_sha
        or prior24_config['inference_and_matcher'] != current_reference_settings
        or prior24_config['conditions'] != json.loads(json.dumps(CONDITIONS))
        or prior24_config['fixed_thresholds'] != {f'{key[0]}_{key[1]}': int(value) for key,value in fixed_thresholds.items()}
        or prior24_config['model_checkpoints'] != {name: model_pointers[name] for name in METHODS[1:3]}):
    raise RuntimeError('Notebook24 fixed recognition provenance differs.')
verify_files(PRIOR24_DIR, prior24_completion['file_sha256'])


In [ ]:
def make_helpers(configuration_sha):
    module = types.ModuleType('frozen_structure_matching_helpers')
    names = ('torch', 'np', 'pd', 'Image', 'Path', 'hashlib', 'json', 'math', 're', 'shutil', 'subprocess', 'zipfile',
        'file_digest', 'json_bytes', 'write_verified', 'training', 'simulator', 'DATA_ROOT', 'pair_config',
        'sim_config', 'pairs', 'probe_records', 'MINDTCT', 'BOZORTH3', 'DEVICE')
    module.__dict__.update({name: globals()[name] for name in names})
    module.evaluation_sha, module.TEMPLATE_DIR, module.model = configuration_sha, TEMPLATE_DIR, None
    exec(helper_code, module.__dict__)
    module.condition_input = dry_condition_input
    return module

def get_unit(family, severity, method):
    if method not in METHODS:
        raise ValueError(f'Unknown method: {method}')
    unit = f'{family}_{severity}_{method}'
    if unit in UNIT_CACHE:
        return UNIT_CACHE[unit]
    reused = method in ('input', 'l1_seed20261110', 'dry_l1_seed20261110')
    if reused:
        provenance = prior24_completion['unit_provenance'][unit]
        directory = Path(provenance['directory'])
        expected_sha = provenance['configuration_sha256']
        if (provenance['condition'] != f'{family}/{severity}' or provenance['method'] != method
                or provenance['source_variant'] != directory.name
                or file_digest(directory / 'completion.json') != provenance['completion_sha256']):
            raise RuntimeError(f'Notebook24 aliased unit provenance changed: {unit}')
        if expected_sha not in reference_helpers:
            reference_helpers[expected_sha] = make_helpers(expected_sha)
        value = reference_helpers[expected_sha].load_unit(directory)
        if value is None:
            raise RuntimeError('Completed notebook24 unit missing.')
        scored, extraction = value
        source_method = 'l1_control' if method == 'l1_seed20261110' else method
        if not extraction['method'].eq(source_method).all() or not extraction['condition'].eq(f'{family}/{severity}').all():
            raise RuntimeError('Notebook24 extraction alias mismatch.')
        value = scored, extraction.assign(method=method)
    else:
        directory = REPORT_DIR / unit
        expected_sha = evaluation_sha
        value = active_helpers.load_unit(directory)
    if value is not None:
        UNIT_CACHE[unit] = value
        UNIT_PROVENANCE[unit] = {'condition': f'{family}/{severity}', 'method': method,
            'origin': 'verified_prior_seed_or_input' if reused else 'notebook26', 'directory': str(directory),
            'configuration_sha256': expected_sha, 'source_variant': directory.name,
            'completion_sha256': file_digest(directory / 'completion.json')}
    return value

def paired_rows(before, after, thresholds, condition, baseline_method, method):
    rows = active_helpers.comparison_rows(before, after, thresholds, condition)
    for row in rows:
        row['baseline_method'], row['method'] = baseline_method, method
        row['baseline_attempt_frr'] = row.pop('input_attempt_frr')
        row['method_attempt_frr'] = row.pop('m0_attempt_frr')
    return rows
def dry_condition_input(record, family, severity):
    if (family, severity) not in CONDITIONS:
        raise ValueError(f'Condition outside compact dry protocol: {family}/{severity}')
    if record['relative_path'] not in validation_paths:
        raise RuntimeError('Dry diagnostic may only open frozen validation sources.')
    dpi = pair_config['source_dpi'][f"{record['dataset']}_{record['db']}"]
    reference = training.normalized_image(str(DATA_ROOT / record['relative_path']), dpi)
    if family == 'identity':
        return reference, reference.copy()
    seed = simulator.case_seed(record['relative_path'], family, severity, sim_config['seed'])
    degraded, _ = simulator.degrade(reference, family, severity, seed)
    return reference, degraded

validation_paths = set(validation['relative_path'])


In [ ]:
adapter_source = '\n\n'.join(inspect.getsource(function) for function in (make_helpers, get_unit, paired_rows, dry_condition_input))
evaluation_config = {'version': 'dry_initialization_validation_v1', 'm0_configuration_sha256': m0_sha,
    'm2_configuration_sha256': m2_sha, 'model_checkpoints': model_pointers,
    'specialist_configuration_sha256': specialist_sha,
    'replication_configuration_sha256': replication_sha,
    'replication_completion_sha256': file_digest(REPLICATION_DIR / 'completion.json'),
    'notebook24_configuration_sha256': prior24_sha,
    'notebook24_completion_sha256': file_digest(PRIOR24_DIR / 'completion.json'),
    'audit12_completion_sha256': file_digest(AUDIT12_DIR / 'completion.json'),
    'specialist_completion_sha256': file_digest(SPECIALIZATION_DIR / 'completion.json'),
    'initialization_seeds': [20261110, 20261111, 20261112], 'best_seed_selection': False,
    'reference_evaluation_sha256': prior_sha, 'reference_completion_sha256': file_digest(PRIOR_DIR / 'completion.json'),
    'fixed_thresholds': {f'{key[0]}_{key[1]}': int(value) for key, value in fixed_thresholds.items()},
    'frozen_helpers_sha256': file_digest(helper_path), 'adapters_sha256': hashlib.sha256(adapter_source.encode()).hexdigest(),
    'pair_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'), 'pair_csv_sha256': pair_config['pair_csv_sha256'],
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'methods': METHODS, 'conditions': CONDITIONS, 'comparisons': COMPARISONS,
    'reference_reuse': True, 'expected_reused_variants': 12, 'expected_new_variants': 8,
    'reference_mismatches': reference_mismatches, 'inference_and_matcher': current_reference_settings,
    'thresholds': 'saved notebook14 original input fixed per db only; no recalibration',
    'scope': 'original dry-like three levels plus identity, development validation; no other families or test',
    'dry_checkpoint_reselection': False, 'training_performed': False}
evaluation_data = json_bytes(evaluation_config)
evaluation_sha = hashlib.sha256(evaluation_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'evaluations' / 'dry_initialization_validation_v1' / evaluation_sha[:20]
CACHE_DIR = LOCAL_ROOT / 'dry_initialization_validation_v1' / evaluation_sha[:20]
TEMPLATE_DIR = CACHE_DIR / 'templates'
for directory in (REPORT_DIR, CACHE_DIR, TEMPLATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
frozen_files = {'configuration.json': evaluation_data, 'frozen_matching_helpers.py.txt': HELPER_SOURCE.encode(),
    'evaluation_adapters.py.txt': adapter_source.encode()}
for name, data in frozen_files.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen dry diagnostic changed: {path}')
    if not path.exists():
        write_verified(path, data)
active_helpers = make_helpers(evaluation_sha)
reference_helpers, UNIT_CACHE, UNIT_PROVENANCE = {}, {}, {}
print('Frozen compact dry evaluation:', evaluation_sha)
print('Reports/resume directory:', REPORT_DIR)


In [ ]:
pending = [(family, severity, method) for family, severity in CONDITIONS for method in METHODS
           if get_unit(family, severity, method) is None]
galleries = None
if pending:
    gallery_rows = []
    for record in gallery_records:
        reference, _ = active_helpers.condition_input(record, 'identity', 'none')
        gallery_rows.append({**record, **active_helpers.extract_png(active_helpers.quantize(reference))})
    galleries = pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR / 'gallery_extraction.csv', galleries)
    if True:
        columns = ['relative_path', 'png_sha256', 'template_sha256', 'minutiae', 'status']
        previous = pd.read_csv(PRIOR_DIR / 'gallery_extraction.csv')[columns].sort_values('relative_path').reset_index(drop=True)
        current = galleries[columns].sort_values('relative_path').reset_index(drop=True)
        if not current.equals(previous):
            raise RuntimeError('Original gallery extraction differs; do not mix reference scores.')
    print('Fixed original galleries prepared:', len(galleries), '| Pending variants:', len(pending), flush=True)
for family, severity in CONDITIONS:
    for method in METHODS:
        unit = f'{family}_{severity}_{method}'
        if get_unit(family, severity, method) is not None:
            print('Reused/resumed verified variant:', unit, flush=True)
            continue
        directory, work = REPORT_DIR / unit, CACHE_DIR / unit
        directory.mkdir(parents=True, exist_ok=True)
        work.mkdir(parents=True, exist_ok=True)
        if method != 'input':
            active_helpers.model = models[method]
        rows = []
        for index, record in enumerate(probe_records, start=1):
            reference, degraded = active_helpers.condition_input(record, family, severity)
            output = degraded if method == 'input' else active_helpers.restore_image(degraded)
            rows.append({**record, **active_helpers.extract_png(active_helpers.quantize(output)),
                **active_helpers.pixel_metrics(reference, degraded, output),
                'corruption_changed_pixel_fraction': float(np.mean(active_helpers.quantize(degraded) != active_helpers.quantize(reference))),
                'condition': f'{family}/{severity}', 'method': method})
            if index % 100 == 0 or index == len(probe_records):
                write_csv(directory / 'extraction_progress.csv', pd.DataFrame(rows))
                print(f'{unit}: extracted/cached {index}/{len(probe_records)} probes', flush=True)
        extraction = pd.DataFrame(rows)
        scored = active_helpers.match_pairs(extraction, galleries, work)
        write_csv(directory / 'scores.csv', scored)
        write_csv(directory / 'extraction.csv', extraction)
        active_helpers.archive_templates(directory / 'templates_and_logs.zip', rows + galleries.to_dict('records'), work)
        marker = {'configuration_sha256': evaluation_sha, 'variant': unit,
            'file_sha256': {name: file_digest(directory / name) for name in ('scores.csv', 'extraction.csv', 'templates_and_logs.zip')}}
        write_verified(directory / 'completion.json', json_bytes(marker))
        get_unit(family, severity, method)
        print('Saved variant:', unit, '| rejected comparisons:', int((~scored['matchable']).sum()), flush=True)
write_csv(REPORT_DIR / 'unit_provenance.csv', pd.DataFrame(UNIT_PROVENANCE.values()))
print('All 20 variants verified; twelve reused, eight new-seed variants, original galleries and thresholds preserved.')

In [ ]:
metric_rows, pixel_rows, paired_changes, threshold_rows, curves = [], [], [], [], []
for family, severity in CONDITIONS:
    condition = f'{family}/{severity}'
    scores = {}
    for method in METHODS:
        scored, extraction = get_unit(family, severity, method)
        scores[method] = scored
        for (dataset, db), images in extraction.groupby(['dataset', 'db'], sort=True):
            pixel_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'probe_images': len(images), 'insufficient_minutiae': int(images['status'].ne('ok').sum()),
                'image_failure_rate': float(images['status'].ne('ok').mean()),
                **{name: float(images[name].mean()) for name in ('float_input_mae', 'float_output_mae', 'float_change_mae',
                    'png_input_mae', 'png_output_mae', 'changed_pixel_fraction')},
                'png_identical_image_fraction': float(images['png_identical_to_input'].mean())})
        for (dataset, db), group in scored.groupby(['dataset', 'db'], sort=True):
            valid_impostors = group.loc[group['matchable'] & group['label'].eq(0), 'score']
            calibrated = None  # No threshold calibration in this stress test.
            available = group.loc[group['matchable'], 'label'].nunique() == 2
            eer, gap = None, None
            if available:
                curve = active_helpers.conditional_curve(group)
                point = curve.loc[(curve['conditional_fmr'] - curve['conditional_fnmr']).abs().idxmin()]
                eer = float((point['conditional_fmr'] + point['conditional_fnmr']) / 2)
                gap = float(abs(point['conditional_fmr'] - point['conditional_fnmr']))
                curves.append(curve.assign(condition=condition, method=method, dataset=dataset, db=db))
            if not available:
                print(f'WARNING: conditional EER unavailable for {condition}/{method}/{dataset}/{db}; all attempts retained.')
            threshold_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'original_fixed_threshold': fixed_thresholds[(dataset, db)], 'condition_calibrated_threshold': calibrated,
                'calibration_status': 'not_performed_saved_original_threshold_only'})
            for policy, threshold in (('original_fixed', fixed_thresholds[(dataset, db)]),):
                if threshold is None:
                    continue
                for view, selected in (('all_impressions', group),
                                       ('byte_distinct_sensitivity', group[group['duplicate_sensitivity_eligible']])):
                    values = active_helpers.metrics_at_threshold(selected, threshold)
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Condition-calibrated validation FMR exceeded 1%.')
                    metric_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, **values,
                        'conditional_eer_discrete_approx': eer if view == 'all_impressions' else None,
                        'eer_crossing_gap': gap if view == 'all_impressions' else None})
    for baseline_method, method in COMPARISONS:
        paired_changes.extend(paired_rows(scores[baseline_method], scores[method], fixed_thresholds, condition, baseline_method, method))
metrics, pixels, paired = pd.DataFrame(metric_rows), pd.DataFrame(pixel_rows), pd.DataFrame(paired_changes)
write_csv(REPORT_DIR / 'validation_metrics.csv', metrics)
write_csv(REPORT_DIR / 'pixel_and_failure_metrics.csv', pixels)
write_csv(REPORT_DIR / 'paired_genuine_changes.csv', paired)
write_csv(REPORT_DIR / 'validation_thresholds.csv', pd.DataFrame(threshold_rows))
write_csv(REPORT_DIR / 'conditional_curves.csv', pd.concat(curves, ignore_index=True))
macro = metrics[metrics['view'].eq('all_impressions')].groupby(['condition', 'method', 'threshold_policy'], sort=True).agg(
    databases=('db', 'size'), macro_attempt_frr=('attempt_frr', 'mean'), macro_attempt_far=('attempt_far', 'mean'),
    macro_conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    macro_conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Macro summaries must include all six databases.')
write_csv(REPORT_DIR / 'macro_operating_points.csv', macro)
write_csv(REPORT_DIR / 'macro_fixed_threshold_metrics.csv', macro[macro['threshold_policy'].eq('original_fixed')])
aggregate_rows = []
scopes = {'dry_like': [f'{family}/{severity}' for family, severity in CONDITIONS if family == 'dry_like']}
for scope, members in scopes.items():
    frame = macro[macro['condition'].isin(members)]
    for (method, policy), group in frame.groupby(['method', 'threshold_policy'], sort=True):
        if len(group) != len(members):
            raise RuntimeError('Incomplete aggregate condition coverage.')
        aggregate_rows.append({'scope': scope, 'method': method, 'threshold_policy': policy,
            'conditions': len(group), **{name: float(group[name].mean()) for name in
                ('macro_attempt_frr', 'macro_attempt_far', 'macro_conditional_fnmr', 'macro_conditional_fmr')},
            'complete_conditional_metrics': bool(group['valid_fnmr_databases'].eq(6).all() and group['valid_fmr_databases'].eq(6).all())})
aggregates = pd.DataFrame(aggregate_rows)
write_csv(REPORT_DIR / 'aggregate_operating_points.csv', aggregates)
print('Equal condition/database weights; only saved original thresholds. This is not independent test performance.')
print(aggregates.to_string(index=False))
print('\nOriginal-image harm/recovery at original fixed thresholds:')
print(paired[paired['condition'].eq('identity/none')].groupby(['baseline_method', 'method'])[['acceptance_recoveries', 'acceptance_harms']].sum().to_string())
print('\nEach method versus input at original fixed thresholds:')
print(paired[paired['baseline_method'].eq('input')].to_string(index=False))
prior_metrics = pd.read_csv(PRIOR_DIR / 'validation_metrics.csv')
identity_checks = []
for method in ('input', 'l1_seed20261110'):
    original_method = 'input' if method == 'input' else 'l1_control'
    for (dataset, db), current in metrics[
            metrics['condition'].eq('identity/none') & metrics['method'].eq(method)
            & metrics['threshold_policy'].eq('original_fixed') & metrics['view'].eq('all_impressions')].groupby(['dataset', 'db']):
        previous = prior_metrics[prior_metrics['condition'].eq('identity/none') & prior_metrics['method'].eq(original_method)
            & prior_metrics['dataset'].eq(dataset) & prior_metrics['db'].eq(db)
            & prior_metrics['threshold_policy'].eq('original_fixed') & prior_metrics['view'].eq('all_impressions')]
        if len(current) != 1 or len(previous) != 1:
            raise RuntimeError('Missing or duplicate identity reference metric.')
        for name in ('threshold', 'genuine_attempts', 'impostor_attempts', 'unmatchable_genuine',
                     'unmatchable_impostor', 'attempt_frr', 'attempt_far', 'conditional_fmr', 'conditional_fnmr'):
            left, right = current.iloc[0][name], previous.iloc[0][name]
            if not ((pd.isna(left) and pd.isna(right)) or (np.isfinite(left) and np.isfinite(right) and abs(left - right) <= 1e-10)):
                raise RuntimeError(f'Fresh identity check differs from notebook14: {method}/{dataset}/{db}/{name}')
        identity_checks.append({'method': method, 'dataset': dataset, 'db': db, 'matches_notebook14': True})
write_csv(REPORT_DIR / 'identity_reference_check.csv', pd.DataFrame(identity_checks))
diagnostic_frames = []
for family, severity in CONDITIONS:
    frame = get_unit(family, severity, 'input')[1].copy()
    if 'corruption_changed_pixel_fraction' not in frame.columns:
        fractions = []
        for record in frame.to_dict('records'):
            reference, degraded = active_helpers.condition_input(record, family, severity)
            if not np.isclose(float(np.abs(degraded - reference).mean()), record['float_input_mae'], rtol=0, atol=1e-7):
                raise RuntimeError('Recomputed input corruption differs from the frozen reference extraction.')
            fractions.append(float(np.mean(active_helpers.quantize(degraded) != active_helpers.quantize(reference))))
        frame['corruption_changed_pixel_fraction'] = fractions
    diagnostic_frames.append(frame[['condition', 'dataset', 'db', 'relative_path', 'float_input_mae',
        'png_input_mae', 'corruption_changed_pixel_fraction']])
diagnostics = pd.concat(diagnostic_frames, ignore_index=True).rename(columns={'corruption_changed_pixel_fraction': 'changed_pixel_fraction'})
write_csv(REPORT_DIR / 'dry_input_diagnostics.csv', diagnostics)
print('Fresh/reused original-reference identity checks passed; the dry specialist is not required to equal mixed L1.')
print('Measure actual intervention strength; these are the original frozen dry levels:')
print(diagnostics.groupby('condition')[['float_input_mae', 'png_input_mae', 'changed_pixel_fraction']].mean().to_string())

comparison_rows = []
scope_frames = [(condition, group.assign(conditions=1)) for condition, group in macro.groupby('condition', sort=True)]
scope_frames += [(scope, group) for scope, group in aggregates.groupby('scope', sort=True)]
for scope, frame in scope_frames:
    if len(frame) != len(METHODS) or set(frame['method']) != set(METHODS):
        raise RuntimeError('Dry summaries must include all five predeclared methods.')
    points = frame.set_index('method')
    for baseline_method, method in COMPARISONS:
        before, after = points.loc[baseline_method], points.loc[method]
        comparison_rows.append({'scope': scope, 'baseline_method': baseline_method, 'method': method,
            'initialization_seed': int(method.removeprefix('dry_l1_seed').removeprefix('l1_seed')), 'conditions': int(after['conditions']),
            **{f'macro_attempt_{rate}': float(after[f'macro_attempt_{rate}']) for rate in ('frr', 'far')},
            **{f'baseline_attempt_{rate}': float(before[f'macro_attempt_{rate}']) for rate in ('frr', 'far')},
            **{f'delta_{rate}_pp': 100 * float(after[f'macro_attempt_{rate}'] - before[f'macro_attempt_{rate}']) for rate in ('frr', 'far')}})
dry_comparisons = pd.DataFrame(comparison_rows)
write_csv(REPORT_DIR / 'dry_operating_point_comparisons.csv', dry_comparisons)
original_harms = paired[paired['condition'].eq('identity/none')].groupby(
    ['baseline_method', 'method'], as_index=False)[['acceptance_recoveries', 'acceptance_harms']].sum()
write_csv(REPORT_DIR / 'original_image_harms.csv', original_harms)
print('Three-initialization dry diagnostic point estimates; descriptive seed variability, not training-population confidence:')
print(dry_comparisons.to_string(index=False))
print('Original-image recoveries/harms, ALL three comparisons:')
print(original_harms.to_string(index=False))

seed_rows = []
for scope, frame in scope_frames:
    values = frame[frame['method'].isin(DRY_METHODS)]
    if len(values) != 3:
        raise RuntimeError('Descriptive seed summary missing a dry initialization.')
    row = {'scope': scope, 'initializations': 3, 'interpretation': 'descriptive initialization-only sample SD, NOT confidence'}
    for rate in ('frr', 'far'):
        points = values[f'macro_attempt_{rate}'].to_numpy()
        row.update({f'mean_{rate}': float(points.mean()), f'sample_sd_{rate}': float(points.std(ddof=1)),
            f'min_{rate}': float(points.min()), f'max_{rate}': float(points.max())})
    seed_rows.append(row)
seed_summary = pd.DataFrame(seed_rows)
write_csv(REPORT_DIR / 'initialization_summary.csv', seed_summary)
print(seed_summary.to_string(index=False))


In [ ]:
REPLICATES, SEED = 2000, 20261110
audit_config = json.loads((AUDIT12_DIR / 'configuration.json').read_text(encoding='utf-8'))
audit_completion = json.loads((AUDIT12_DIR / 'completion.json').read_text(encoding='utf-8'))
helper_path12 = AUDIT12_DIR / 'analysis_helpers.py.txt'
if (audit_completion['status'] != 'validation_tradeoff_audit_complete'
        or audit_completion['configuration_sha256'] != file_digest(AUDIT12_DIR / 'configuration.json')
        or audit_config['replicates'] != REPLICATES or audit_config['seed'] != SEED
        or file_digest(helper_path12) != audit_config['helper_sha256']
        or file_digest(helper_path12) != audit_completion['file_sha256']['analysis_helpers.py.txt']):
    raise RuntimeError('Frozen audit12 paired-bootstrap helper mismatch.')
bootstrap_source = helper_path12.read_bytes().decode('utf-8')
bootstrap_namespace = dict(globals())
exec(compile(bootstrap_source, str(helper_path12), 'exec'), bootstrap_namespace)
canonical = get_unit('identity', 'none', 'input')[0].sort_values('pair_id').reset_index(drop=True)
databases = sorted(set(zip(canonical['dataset'], canonical['db'])))
finger_keys, bootstrap_weights = {}, {}
for dataset, db in databases:
    keys = sorted(canonical.loc[canonical['dataset'].eq(dataset) & canonical['db'].eq(db), 'probe_identity'].unique())
    if len(keys) < 2:
        raise RuntimeError('Insufficient probe-finger clusters.')
    finger_keys[(dataset, db)] = keys
    seed = int.from_bytes(hashlib.sha256(f'{SEED}|probe_cluster|{dataset}|{db}'.encode()).digest()[:8], 'big')
    weights = np.random.default_rng(seed).multinomial(len(keys), np.full(len(keys), 1 / len(keys)), size=REPLICATES)
    if hashlib.sha256(weights.astype('<i8').tobytes()).hexdigest() != audit_config['draw_sha256'][f'{dataset}_{db}']:
        raise RuntimeError('Audit12 paired cluster draw hash differs.')
    bootstrap_weights[(dataset, db)] = weights
bootstrap_namespace.update(databases=databases, finger_keys=finger_keys, bootstrap_weights=bootstrap_weights)
interval_row = bootstrap_namespace['interval_row']
pieces, interval_rows = {}, []
for family, level in CONDITIONS:
    condition = f'{family}/{level}'
    units = {method: get_unit(family, level, method)[0].sort_values('pair_id').reset_index(drop=True) for method in METHODS}
    for method, unit in units.items():
        metadata = ['pair_id', 'dataset', 'db', 'label', 'probe_identity', 'probe_path', 'gallery_path']
        if not unit[metadata].equals(canonical[metadata]):
            raise RuntimeError(f'Bootstrap pair alignment/metadata mismatch: {condition}/{method}')
    for baseline, method in COMPARISONS:
        value = bootstrap_namespace['bootstrap_pair'](units[baseline], units[method], fixed_thresholds, fixed_thresholds)
        pieces[(condition, baseline, method)] = value
        interval_rows.append(interval_row(condition, 'original_fixed', baseline, method, value))
for baseline, method in COMPARISONS:
    selected = [pieces[(f'dry_like/{level}', baseline, method)] for level in ('mild', 'moderate', 'severe')]
    value = {rate: {'delta': float(np.mean([v[rate]['delta'] for v in selected])),
        'replicates': np.mean([v[rate]['replicates'] for v in selected], axis=0)} for rate in ('frr','far')}
    pieces[('dry_like', baseline, method)] = value
    interval_rows.append(interval_row('dry_like', 'original_fixed', baseline, method, value))
for scope in ['identity/none', 'dry_like/mild', 'dry_like/moderate', 'dry_like/severe', 'dry_like']:
    for baseline in ('input', 'l1_seed20261110'):
        selected = [pieces[(scope, baseline, method)] for method in DRY_METHODS]
        value = {rate: {'delta': float(np.mean([v[rate]['delta'] for v in selected])),
            'replicates': np.mean([v[rate]['replicates'] for v in selected], axis=0)} for rate in ('frr','far')}
        interval_rows.append(interval_row(scope, 'original_fixed', baseline, 'mean_of_three_fixed_dry_checkpoints', value))
intervals = pd.DataFrame(interval_rows)
write_csv(REPORT_DIR / 'paired_probe_finger_intervals.csv', intervals)
write_verified(REPORT_DIR / 'bootstrap_helpers.py.txt', bootstrap_source.encode())
bootstrap_provenance = {'replicates': REPLICATES, 'seed': SEED,
    'audit12_configuration_sha256': file_digest(AUDIT12_DIR / 'configuration.json'),
    'helper_sha256': file_digest(helper_path12), 'draw_sha256': audit_config['draw_sha256'],
    'scope': 'pointwise conditional on fixed galleries/thresholds/checkpoints and three fixed seeds; not training-population confidence'}
write_verified(REPORT_DIR / 'bootstrap_provenance.json', json_bytes(bootstrap_provenance))
print('Conditional paired cluster intervals; FRR and FAR deltas in percentage points:')
print(intervals[intervals['scope'].isin(['dry_like','identity/none'])].to_string(index=False))


In [ ]:
record = probe_records[0]
figure, axes = plt.subplots(len(CONDITIONS), len(METHODS) + 1, figsize=(15, 2.4 * len(CONDITIONS)))
for row, (family, severity) in enumerate(CONDITIONS):
    reference, degraded = active_helpers.condition_input(record, family, severity)
    images = [('input', degraded)]
    for method in METHODS[1:]:
        active_helpers.model = models[method]
        images.append((method, active_helpers.restore_image(degraded)))
    images.append(('reference', reference))
    for column, (title, image) in enumerate(images):
        axes[row, column].imshow(active_helpers.quantize(image), cmap='gray', vmin=0, vmax=255)
        axes[row, column].set_title(f'{family}/{severity} | {title}', fontsize=7)
        axes[row, column].axis('off')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'full_image_preview.png', dpi=140)
plt.show()
plt.close(figure)
if len(UNIT_PROVENANCE) != len(CONDITIONS) * len(METHODS):
    raise RuntimeError('Missing method/condition variants at completion.')
files = [*frozen_files, 'unit_provenance.csv', 'gallery_extraction.csv', 'validation_metrics.csv',
    'pixel_and_failure_metrics.csv', 'paired_genuine_changes.csv', 'validation_thresholds.csv', 'conditional_curves.csv',
    'macro_operating_points.csv', 'macro_fixed_threshold_metrics.csv', 'aggregate_operating_points.csv', 'full_image_preview.png', 'identity_reference_check.csv', 'dry_input_diagnostics.csv', 'dry_operating_point_comparisons.csv', 'original_image_harms.csv', 'initialization_summary.csv',
    'paired_probe_finger_intervals.csv', 'bootstrap_helpers.py.txt', 'bootstrap_provenance.json']
completion = {'status': 'dry_initialization_validation_complete', 'configuration_sha256': evaluation_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files}, 'unit_provenance': UNIT_PROVENANCE,
    'methods': METHODS, 'variants_available': len(UNIT_PROVENANCE), 'best_seed_selected': False, 'initialization_seeds': [20261110, 20261111, 20261112], 'pairs_per_variant': len(pairs),
    'reference_reuse': True, 'reused_variants': 12, 'new_variants': 8, 'independent_test_improvement_verified': False,
    'training_performed': False, 'new_thresholds_calibrated': False, 'external_test_evaluated': False,
    'interpretation': 'three fixed dry checkpoints versus input and fixed mixed seed20261110; descriptive seed SD and conditional pointwise probe-cluster intervals, not training-population confidence; no isolated gradient conflict, routing, novelty or test claims'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('DRY INITIALIZATION VALIDATION COMPLETE:', REPORT_DIR)
print('Send per-seed points, initialization summary, paired intervals and original-image harms. Do not choose a best seed, reselect epochs or open test.')
write_verified(REPORT_DIR.parent / 'completed_evaluation.json', json_bytes({
    'directory': str(REPORT_DIR), 'configuration_sha256': evaluation_sha,
    'completion_sha256': file_digest(REPORT_DIR / 'completion.json')}))
